In [2]:
import polars as pl

df = pl.scan_csv("~/google-transparency-report/google-websearch-copyright-removals/requests.csv.zst")

In [6]:
df.collect_schema()

Schema([('Request ID', Int64),
        ('Date', String),
        ('Lumen URL', String),
        ('Copyright owner ID', Int64),
        ('Copyright owner name', String),
        ('Reporting organization ID', Int64),
        ('Reporting organization name', String),
        ('URLs removed', Int64),
        ("URLs that were not in Google's search index", Int64),
        ('URLs for which we took no action', Int64),
        ('URLs pending review', Int64),
        ('From Abuser', Boolean)])

In [9]:
df.select(pl.col("URLs removed")).sum().collect()

URLs removed
i64
15096274436


In [10]:
print(df.select(pl.col("URLs removed").sum()).explain())

SELECT [col("URLs removed").sum()]
  Csv SCAN [/home/dinhhuy2010/google-transparency-report/google-websearch-copyright-removals/requests.csv.zst]
  PROJECT 1/12 COLUMNS
  ESTIMATED ROWS: 26854178


In [3]:
df.sink_parquet(
    "/home/dinhhuy2010/google-transparency-report/google-websearch-copyright-removals/requests.parquet",
    compression="zstd",
    compression_level=9,
    row_group_size=131_072,
)

In [8]:
!eza --icons -la ~/google-transparency-report/google-websearch-copyright-removals

.rw-r--r-- 2.3k dinhhuy2010 20 Oct 12:43  README.txt
.rw-r--r-- 1.2G dinhhuy2010 21 Oct 20:39  requests.csv.zst
.rw-r--r-- 1.2G dinhhuy2010 23 Oct 22:57  requests.parquet


In [9]:
pqdf = pl.scan_parquet("~/google-transparency-report/google-websearch-copyright-removals/requests.parquet")
pqdf.select(pl.col("URLs removed")).sum().collect()

URLs removed
i64
15096274436


In [13]:
import polars as pl

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(1000)

polars.config.Config

In [15]:
# Historical growth of copyright removals
yearly = (
    pqdf.with_columns(
        pl.col("Date").str.slice(0, 4).alias("Year")
    )
    .group_by("Year")
    .agg(
        pl.len().alias("requests"),
        pl.col("URLs removed").sum().alias("removed"),
    )
    .sort("Year")
    .collect()
)
yearly

Year,requests,removed
str,u32,i64
"""2011""",35489,3192256
"""2012""",427969,54140640
"""2013""",479930,221848808
"""2014""",830891,320632396
"""2015""",1190423,519728588
"""2016""",1251752,914265541
"""2017""",1298758,692823611
"""2018""",1584770,226363219
"""2019""",1484751,185786094


In [17]:
top_owners = (
    pqdf.group_by(
        "Copyright owner ID",
        "Copyright owner name"
    )
    .agg(
        pl.len().alias("requests"),
        pl.col("URLs removed").sum().alias("removed")
    )
    .sort("removed", descending=True)
    .head(20)
    .collect()
)

top_owners

Copyright owner ID,Copyright owner name,requests,removed
i64,str,u32,i64
22091,"""penguin random house llc""",300204,1945290927
197206,"""harpercollins publishers llc""",192356,1361235963
428120,"""hachette""",117844,797506613
22091,"""BPI LTD MEMBER COMPANIES""",358075,570109656
29149,"""Editis""",128776,455271284
70158,"""mg premium ltd""",60194,451292558
79381,"""taylor & francis group""",58136,440440840
70158,"""MG Premium Ltd""",97508,344870280
288689,"""kensington publishing corp.""",33440,288559747


In [18]:
import time

start = time.perf_counter()

result = pqdf.group_by(
    "Copyright owner ID",
    "Copyright owner name"
).agg(
    pl.len().alias("requests"),
    pl.col("URLs removed").sum().alias("removed"),
    pl.col("URLs removed").mean().alias("mean"),
    pl.col("URLs removed").max().alias("largest_request"),
    pl.col("URLs removed").median().alias("median"),
).sort("removed", descending=True).head(20).collect()

print(f"Elapsed: {time.perf_counter() - start:.4f}s")
result

Elapsed: 12.7624s


Copyright owner ID,Copyright owner name,requests,removed,mean,largest_request,median
i64,str,u32,i64,f64,i64,f64
22091,"""penguin random house llc""",300204,1945290927,6479.89676,19565,9205.0
197206,"""harpercollins publishers llc""",192356,1361235963,7076.649353,19271,9288.0
428120,"""hachette""",117844,797506613,6767.477453,19965,9314.0
22091,"""BPI LTD MEMBER COMPANIES""",358075,570109656,1592.151521,18658,1310.0
29149,"""Editis""",128776,455271284,3535.373703,5000,4701.0
70158,"""mg premium ltd""",60194,451292558,7497.301359,12005,8683.0
79381,"""taylor & francis group""",58136,440440840,7576.043071,12420,9337.0
70158,"""MG Premium Ltd""",97508,344870280,3536.840875,24745,2000.0
288689,"""kensington publishing corp.""",33440,288559747,8629.179037,17684,9457.0
